In [18]:
import sys, torch
print('python_executable:', sys.executable)
print('torch_version:', torch.__version__)
print('torch_cuda_runtime:', torch.version.cuda)
print('cuda_available:', torch.cuda.is_available())
print('gpu_count:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('gpu_name:', torch.cuda.get_device_name(0))

python_executable: c:\Users\gianf\AppData\Local\Programs\Python\Python313\python.exe
torch_version: 2.11.0+cu128
torch_cuda_runtime: 12.8
cuda_available: True
gpu_count: 1
gpu_name: NVIDIA GeForce RTX 3080


Imports

In [19]:
import pandas as pd
import numpy as np
import torch
from umap import UMAP
from sentence_transformers import SentenceTransformer
from hdbscan import HDBSCAN
from bertopic import BERTopic
from bertopic.vectorizers import OnlineCountVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

Load Data

In [20]:
df = pd.read_csv('Data/finnhub_model_input.csv')
print('Loaded: Data/finnhub_model_input.csv')
print('Shape:', df.shape)
display(df.head())

Loaded: Data/finnhub_model_input.csv
Shape: (5006, 5)


,stock,date,date_only,headline,headline_raw
0,ING,2024-09-17 15:00:55,2024-09-17,What are share buybacks?,What are share buybacks?
1,SMFG,2025-04-02 03:26:24,2025-04-02,PBOC Seen Deploying Stimulus Soon on Tariff Ri...,PBOC Seen Deploying Stimulus Soon on Tariff Ri...
2,BACHY,2025-04-02 03:26:24,2025-04-02,PBOC Seen Deploying Stimulus Soon on Tariff Ri...,PBOC Seen Deploying Stimulus Soon on Tariff Ri...
3,BACHY,2025-04-02 07:33:46,2025-04-02,"Across Chinese Fintech And Bank Stocks, See Wh...","Across Chinese Fintech And Bank Stocks, See Wh..."
4,SMFG,2025-04-02 12:41:01,2025-04-02,Japan's __COMPANY__ Plans Stablecoin Developme...,Japan's SMBC Plans Stablecoin Development With...


In [21]:
# 1. Parse date and prepare chronological data
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.dropna(subset=['date', 'headline']).sort_values('date').reset_index(drop=True)

# Deduplicate semantically identical headlines (topic modeling should not overcount copies across tickers)
before_dedup = len(df)
df = df.drop_duplicates(subset=['headline']).sort_values('date').reset_index(drop=True)
after_dedup = len(df)

# 2. Define split proportions and embargo
SPLIT_TRAIN = 0.70
SPLIT_VAL = 0.25
SPLIT_TEST = 0.05

# 3. Calculate date-based boundaries (keep same day together)
unique_dates = np.array(sorted(df['date'].dt.floor('D').unique()))
n_dates = len(unique_dates)

train_end_idx = int(SPLIT_TRAIN * n_dates)
val_end_idx = int((SPLIT_TRAIN + SPLIT_VAL) * n_dates)

# 4. Create masks with embargo
day_code, _ = pd.factorize(df['date'].dt.floor('D'), sort=True)
mask_train = day_code < train_end_idx
mask_val = (day_code >= (train_end_idx)) & (day_code < val_end_idx)
mask_test = day_code >= (val_end_idx)

# 5. Extract splits
train_df = df[mask_train].copy()
val_df = df[mask_val].copy()
test_df = df[mask_test].copy()

# Final lists for the model
train_docs, train_timestamps = train_df['headline'].tolist(), train_df['date'].tolist()
val_docs, val_timestamps = val_df['headline'].tolist(), val_df['date'].tolist()
test_docs, test_timestamps = test_df['headline'].tolist(), test_df['date'].tolist()

print(f'Deduplication: {before_dedup} -> {after_dedup} rows')
print(f"Train size: {len(train_docs)}")
print(f"Validation size: {len(val_docs)}")
print(f"Test size: {len(test_docs)}")
print('Date ranges:')
print(f"  Train: {train_df['date'].min()} -> {train_df['date'].max()}")
print(f"  Val:   {val_df['date'].min()} -> {val_df['date'].max()}")
print(f"  Test:  {test_df['date'].min()} -> {test_df['date'].max()}")

Deduplication: 5006 -> 4300 rows
Train size: 1967
Validation size: 1178
Test size: 1155
Date ranges:
  Train: 2024-09-17 15:00:55 -> 2025-12-11 23:21:43
  Val:   2025-12-12 04:00:00 -> 2026-03-10 23:14:51
  Test:  2026-03-11 07:10:44 -> 2026-03-28 19:00:00


Prepare Models

In [22]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Torch CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

# Add placeholder tokens to stopwords so they do not dominate topics
placeholder_stopwords = {'__ticker__', '__company__', 'ticker', 'company'}
model_stopwords = sorted(set(ENGLISH_STOP_WORDS).union(placeholder_stopwords))

sentence_model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
train_embeddings = sentence_model.encode(
    train_docs,
    show_progress_bar=True,
    batch_size=64,
    convert_to_numpy=True
)
print(f'Embeddings computed on device: {device}')

Torch CUDA available: True
GPU: NVIDIA GeForce RTX 3080


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1088.83it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1088.83it/s, Materializing param=pooler.dense.weight]                
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 31/31 [00:00<00:00, 59.22it/s]

Embeddings computed on device: cuda


Train BERTopic

In [23]:
print("Training BERTopic model on training split...\n")

umap_model = UMAP(n_neighbors=15, n_components=10, metric='cosine')
hdbscan_model = HDBSCAN(min_cluster_size=10, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
vectorizer_model = OnlineCountVectorizer(stop_words=model_stopwords)

topic_model = BERTopic(
  embedding_model=sentence_model,
  umap_model=umap_model,
  hdbscan_model=hdbscan_model,
  vectorizer_model=vectorizer_model,
  calculate_probabilities=True,
  verbose=True
)

# Train BERTopic only on train split
topics_train, probs_train = topic_model.fit_transform(train_docs, embeddings=train_embeddings)

print("\nModel training complete!")

2026-04-07 21:11:43,303 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


Training BERTopic model on training split...



2026-04-07 21:11:46,605 - BERTopic - Dimensionality - Completed ✓
2026-04-07 21:11:46,607 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-07 21:11:46,607 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-07 21:11:46,830 - BERTopic - Cluster - Completed ✓
2026-04-07 21:11:46,834 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-07 21:11:46,830 - BERTopic - Cluster - Completed ✓
2026-04-07 21:11:46,834 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-07 21:11:46,871 - BERTopic - Representation - Completed ✓
2026-04-07 21:11:46,871 - BERTopic - Representation - Completed ✓



Model training complete!


Get Topic Information

In [24]:
# Get topic information
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,677,-1_bank_street_wall_2025,"[bank, street, wall, 2025, week, billion, says...",[__TICKER__ sees up to $1.6 billion in loss on...
1,0,82,0_commentary_fund_q1_international,"[commentary, fund, q1, international, 2025, q2...",[Thornburg Better World International Fund Q1 ...
2,1,76,1_china_chinese_gold_stimulus,"[china, chinese, gold, stimulus, pboc, trade, ...","[Wall Street Sees Higher China Growth, Less St..."
3,2,65,2_valuation_recent_assessing_share,"[valuation, recent, assessing, share, dbk, pri...",[Banco __COMPANY__ (BME: __TICKER__ ): Assessi...
4,3,58,3_dividend_portfolio_yield_10,"[dividend, portfolio, yield, 10, stocks, high,...",[My Dividend Stock Portfolio: New July Dividen...
5,4,51,4_stablecoin_crypto_banks_major,"[stablecoin, crypto, banks, major, swift, bloc...",[__COMPANY__ becomes first major bank to launc...
6,5,48,5_update_afternoon_sector_financial,"[update, afternoon, sector, financial, stocks,...",[Sector Update: Financial Stocks Rise Late Aft...
7,6,47,6_buyback_share_programme_voting,"[buyback, share, programme, voting, rights, sh...",[__COMPANY__ share buyback programme - Declara...
8,7,42,7_japan_japanese_megabanks_bonds,"[japan, japanese, megabanks, bonds, higher, el...",[Shares of US-listed Japanese stocks are tradi...
9,8,34,8_ai_technology_driven_innovation,"[ai, technology, driven, innovation, joins, in...",[__TICKER__ joins the Massachusetts Institute ...


Visualize Topics

In [25]:
topic_model.visualize_documents(train_docs, embeddings=train_embeddings)

In [26]:
fig = topic_model.visualize_heatmap()
fig.show()

In [27]:
hierarchical_topics = topic_model.hierarchical_topics(train_docs)
fig = topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)
fig.show()

print("💡 This shows the hierarchical structure of topics")

100%|██████████| 46/46 [00:00<00:00, 594.13it/s]



💡 This shows the hierarchical structure of topics


In [28]:
import plotly.express as px

topics_over_time = topic_model.topics_over_time(
    train_docs,
    train_timestamps,
    nr_bins=20
 )

tot = topics_over_time.copy()
tot = tot[tot['Topic'] != -1].copy()
tot['Timestamp'] = pd.to_datetime(tot['Timestamp']).dt.to_period('M').dt.to_timestamp()

topic_labels = topic_model.get_topic_info()[['Topic', 'Name']].copy()
topic_labels = topic_labels[topic_labels['Topic'] != -1]
tot = tot.merge(topic_labels, on='Topic', how='left')
tot['TopicLabel'] = tot['Topic'].astype(str) + ': ' + tot['Name'].fillna('')

stacked = (
    tot.groupby(['Timestamp', 'TopicLabel'], as_index=False)['Frequency']
       .sum()
       .sort_values(['Timestamp', 'Frequency'], ascending=[True, False])
)

fig = px.bar(
    stacked,
    x='Timestamp',
    y='Frequency',
    color='TopicLabel',
    title='Topic Frequency Over Time (Stacked)',
    labels={'Timestamp': 'Month', 'Frequency': 'Document Count', 'TopicLabel': 'Topic'}
)
fig.update_layout(
    barmode='stack',
    xaxis_title='Month',
    yaxis_title='Document Count',
    legend_title='Topic',
    hovermode='x unified'
 )
fig.show()

13it [00:00, 50.85it/s]



Preparing Validation

In [29]:
bertopic_topics = [
    [word for word, _ in topic_model.get_topic(t)]
    for t in topic_model.get_topics().keys() if t != -1
    if topic_model.get_topic(t) is not None
 ]

# Project validation documents into trained topic space
val_topics, val_probs = topic_model.transform(val_docs)

print("Validation transform complete (test set remains untouched).")

Batches: 100%|██████████| 37/37 [00:00<00:00, 59.11it/s]
2026-04-07 21:11:51,308 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
Batches: 100%|██████████| 37/37 [00:00<00:00, 59.11it/s]
2026-04-07 21:11:51,308 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-04-07 21:11:54,755 - BERTopic - Dimensionality - Completed ✓
2026-04-07 21:11:54,756 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-04-07 21:11:54,755 - BERTopic - Dimensionality - Completed ✓
2026-04-07 21:11:54,756 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-04-07 21:11:54,817 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2026-04-07 21:11:54,817 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2026-04-07 21:11:54,957 - BERTopic - Probabilities - Completed ✓
2026-04-07 21:11:54,958 - BERTopic - Cluster - Completed ✓
2026-04-07 21:11:54,957 - BERTopic - P

Validation transform complete (test set remains untouched).


In [30]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora import Dictionary
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import silhouette_score
import nltk
import warnings
from nltk.corpus import stopwords

# Download required NLTK data
nltk.download('stopwords', quiet=True)

# Numerical stability controls
COHERENCE_EPS = 1e-8
COHERENCE_MIN_TEXTS = 20
COHERENCE_MIN_VOCAB = 30

# Build a tokenizer aligned with BERTopic vocabulary (supports unigrams + bigrams)
stop_words = list(model_stopwords) if 'model_stopwords' in globals() else list(stopwords.words('english'))
coherence_vectorizer = CountVectorizer(
    stop_words=stop_words,
    ngram_range=(1, 2),
    token_pattern=r'(?u)\b\w\w+\b'
)
coherence_analyzer = coherence_vectorizer.build_analyzer()

def preprocess_documents(doc_list):
    tokenized = [coherence_analyzer(str(doc).lower()) for doc in doc_list]
    return [tokens for tokens in tokenized if len(tokens) >= 2]

processed_train_docs = preprocess_documents(train_docs)
processed_val_docs = preprocess_documents(val_docs)

# Dictionary from train split only
id2word = Dictionary(processed_train_docs)

def topic_diversity(topics):
    all_words = [word for topic in topics for word in topic]
    unique_words = set(all_words)
    return len(unique_words) / len(all_words) if len(all_words) > 0 else 0.0

def _filter_topics_for_dictionary(topics, dictionary, min_words_per_topic=3):
    vocab = dictionary.token2id
    filtered = []
    for topic in topics:
        cleaned = []
        for term in topic:
            token = str(term).strip().lower()
            if token in vocab:
                cleaned.append(token)
        cleaned = list(dict.fromkeys(cleaned))
        if len(cleaned) >= min_words_per_topic:
            filtered.append(cleaned)
    return filtered

def safe_silhouette_score(
    embeddings,
    labels,
    outlier_label=-1,
    metric='cosine',
    min_cluster_size_for_metric=2,
    min_points_for_metric=10,
    drop_singletons=True
):
    if embeddings is None or labels is None:
        return np.nan

    labels_arr = np.asarray(labels)
    embeddings_arr = np.asarray(embeddings)

    if embeddings_arr.ndim != 2 or labels_arr.ndim != 1:
        return np.nan
    if embeddings_arr.shape[0] != labels_arr.shape[0]:
        return np.nan

    finite_mask = np.isfinite(labels_arr.astype(float, copy=False))
    valid_mask = (labels_arr != outlier_label) & finite_mask
    if valid_mask.sum() < min_points_for_metric:
        return np.nan

    labels_valid = labels_arr[valid_mask]
    emb_valid = embeddings_arr[valid_mask]

    if drop_singletons:
        unique_labels, counts = np.unique(labels_valid, return_counts=True)
        keep_labels = unique_labels[counts >= min_cluster_size_for_metric]
        if len(keep_labels) < 2:
            return np.nan
        keep_mask = np.isin(labels_valid, keep_labels)
        labels_valid = labels_valid[keep_mask]
        emb_valid = emb_valid[keep_mask]

    unique_labels, counts = np.unique(labels_valid, return_counts=True)
    if len(unique_labels) < 2:
        return np.nan
    if np.any(counts < min_cluster_size_for_metric):
        return np.nan
    if len(labels_valid) < max(min_points_for_metric, len(unique_labels) + 1):
        return np.nan

    try:
        value = float(silhouette_score(emb_valid, labels_valid, metric=metric))
        return value if np.isfinite(value) else np.nan
    except Exception:
        return np.nan

def coherence_score(topics, texts, dictionary, coherence_type='c_v', jitter=COHERENCE_EPS):
    clean_texts = [t for t in texts if isinstance(t, list) and len(t) >= 2]
    if len(clean_texts) < COHERENCE_MIN_TEXTS or len(dictionary) < COHERENCE_MIN_VOCAB:
        return np.nan if jitter <= 0 else float(jitter)

    local_dict = Dictionary(clean_texts)
    local_dict.filter_extremes(no_below=2, no_above=0.95)
    local_dict.compactify()
    if len(local_dict) < 20:
        return np.nan if jitter <= 0 else float(jitter)

    filtered_topics = _filter_topics_for_dictionary(topics, local_dict, min_words_per_topic=3)
    if len(filtered_topics) < 2:
        return np.nan if jitter <= 0 else float(jitter)

    def _compute_value(kind):
        if kind == 'u_mass':
            corpus = [local_dict.doc2bow(doc) for doc in clean_texts]
            cm = CoherenceModel(
                topics=filtered_topics,
                corpus=corpus,
                dictionary=local_dict,
                coherence='u_mass'
            )
        else:
            cm = CoherenceModel(
                topics=filtered_topics,
                texts=clean_texts,
                dictionary=local_dict,
                coherence=kind
            )
        return float(cm.get_coherence())

    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', category=RuntimeWarning, module='gensim')
        with np.errstate(divide='ignore', invalid='ignore'):
            try:
                value = _compute_value(coherence_type)
                if np.isfinite(value):
                    return float(value)
            except Exception:
                pass

            try:
                fallback_value = _compute_value('u_mass')
                if np.isfinite(fallback_value):
                    return float(fallback_value)
            except Exception:
                pass

    return np.nan if jitter <= 0 else float(jitter)

cv_train = coherence_score(bertopic_topics, processed_train_docs, id2word, 'c_v')
cv_val = coherence_score(bertopic_topics, processed_val_docs, id2word, 'c_v')
div_train = topic_diversity(bertopic_topics)
val_outlier_ratio = float(np.mean(np.array(val_topics) == -1)) if len(val_topics) > 0 else np.nan
valid_topic_count = len(_filter_topics_for_dictionary(bertopic_topics, id2word, min_words_per_topic=3))

val_embeddings_for_metrics = sentence_model.encode(
    val_docs,
    show_progress_bar=True,
    batch_size=64,
    convert_to_numpy=True
)
sil_val = safe_silhouette_score(val_embeddings_for_metrics, val_topics)
sil_val = 0.0 if not np.isfinite(sil_val) else sil_val

print(f"BERTopic Coherence C_v (train): {cv_train:.4f}")
print(f"BERTopic Coherence C_v (val):   {cv_val:.4f}")
print(f"BERTopic Topic Diversity (train): {div_train:.4f}")
print(f"Validation outlier ratio (-1 topics): {val_outlier_ratio:.4f}")
print(f"Validation silhouette (cosine, no outliers): {sil_val:.4f}")
print(f"Valid topic count for coherence: {valid_topic_count}")

Batches: 100%|██████████| 19/19 [00:00<00:00, 41.40it/s]

BERTopic Coherence C_v (train): 0.5307
BERTopic Coherence C_v (val):   0.4362
BERTopic Topic Diversity (train): 0.8660
Validation outlier ratio (-1 topics): 0.5272
Validation silhouette (cosine, no outliers): 0.1320
Valid topic count for coherence: 47


Hyperparameter tuning with rolling time-series CV (train+val only, fixed seed) and bucket-wise reporting

In [34]:
from itertools import product
import time

# -------------------- Time-series CV controls (fixed seed for model selection) --------------------
N_FOLDS = 3  # rolling/expanding buckets before test
RANDOM_SEED = 42

# Build tuning pool from train + validation only (test remains untouched)
tune_df = pd.concat([train_df, val_df], axis=0).sort_values('date').reset_index(drop=True)
tune_docs = tune_df['headline'].tolist()
tune_timestamps = tune_df['date'].tolist()

# Precompute embeddings for the full tuning pool once
tune_embeddings = sentence_model.encode(
    tune_docs,
    show_progress_bar=True,
    batch_size=64,
    convert_to_numpy=True
)

def sanitize_metric(value, fallback=np.nan):
    try:
        v = float(value)
    except Exception:
        return fallback
    return v if np.isfinite(v) else fallback

def singleton_ratio_from_labels(labels, outlier_label=-1):
    labels_arr = np.asarray(labels)
    labels_arr = labels_arr[labels_arr != outlier_label]
    if labels_arr.size == 0:
        return np.nan
    counts = pd.Series(labels_arr).value_counts()
    if counts.empty:
        return np.nan
    return float((counts == 1).mean())

def make_expanding_folds(df_dates, n_folds=3, embargo_n=1):
    unique_days = np.array(sorted(pd.to_datetime(df_dates).dt.floor('D').unique()))
    n_days = len(unique_days)
    edges = np.linspace(0, n_days, n_folds + 2, dtype=int)
    day_code, _ = pd.factorize(pd.to_datetime(df_dates).dt.floor('D'), sort=True)

    folds = []
    for fold_idx in range(n_folds):
        train_end = edges[fold_idx + 1]
        val_start = train_end + embargo_n
        val_end = edges[fold_idx + 2]

        if train_end <= 0 or val_start >= val_end:
            continue

        train_mask = day_code < train_end
        val_mask = (day_code >= val_start) & (day_code < val_end)

        train_idx = np.where(train_mask)[0]
        val_idx = np.where(val_mask)[0]

        if len(train_idx) == 0 or len(val_idx) == 0:
            continue

        folds.append({
            'fold': fold_idx + 1,
            'train_idx': train_idx,
            'val_idx': val_idx,
            'train_start': unique_days[0],
            'train_end': unique_days[train_end - 1],
            'val_start': unique_days[val_start],
            'val_end': unique_days[val_end - 1]
        })

    return folds

folds = make_expanding_folds(tune_df['date'], n_folds=N_FOLDS)
print(f'Generated {len(folds)} rolling folds (train+val pool only).')
for fold in folds:
    print(
        f"Fold {fold['fold']}: train {fold['train_start']} -> {fold['train_end']} | "
        f"val {fold['val_start']} -> {fold['val_end']}"
    )

# Expanded UMAP/HDBSCAN ranges to reduce outliers/singletons
param_grid = {
    'n_neighbors': [15, 30, 50],
    'n_components': [5, 10],
    'min_cluster_size': [5, 8, 10, 20],
    'min_samples': [1, 3, 5],
    'ngram_range': [(1, 1), (1, 2)]
}

search_space = list(product(
    param_grid['n_neighbors'],
    param_grid['n_components'],
    param_grid['min_cluster_size'],
    param_grid['min_samples'],
    param_grid['ngram_range']
))

print(f'Running {len(search_space)} parameter combinations x {len(folds)} folds (fixed seed={RANDOM_SEED})')

cv_rows = []

for trial, (n_neighbors, n_components, min_cluster_size, min_samples, ngram_range) in enumerate(search_space, start=1):
    print(
        f"[Trial {trial}/{len(search_space)}] n_neighbors={n_neighbors}, n_components={n_components}, "
        f"min_cluster_size={min_cluster_size}, min_samples={min_samples}, ngram_range={ngram_range}"
    )

    for fold in folds:
        train_idx = fold['train_idx']
        val_idx = fold['val_idx']

        fold_train_docs = [tune_docs[i] for i in train_idx]
        fold_val_docs = [tune_docs[i] for i in val_idx]
        fold_train_embeddings = tune_embeddings[train_idx]
        fold_val_embeddings = tune_embeddings[val_idx]

        processed_train_fold = preprocess_documents(fold_train_docs)
        processed_val_fold = preprocess_documents(fold_val_docs)
        id2word_fold = Dictionary(processed_train_fold)

        start = time.perf_counter()

        umap_model_i = UMAP(
            n_neighbors=n_neighbors,
            n_components=n_components,
            metric='cosine',
            random_state=RANDOM_SEED
        )
        hdbscan_model_i = HDBSCAN(
            min_cluster_size=min_cluster_size,
            min_samples=min_samples,
            metric='euclidean',
            cluster_selection_method='eom',
            prediction_data=True,
            core_dist_n_jobs=-1
        )
        vectorizer_model_i = OnlineCountVectorizer(stop_words=model_stopwords, ngram_range=ngram_range)

        topic_model_i = BERTopic(
            embedding_model=sentence_model,
            umap_model=umap_model_i,
            hdbscan_model=hdbscan_model_i,
            vectorizer_model=vectorizer_model_i,
            calculate_probabilities=False,
            verbose=False
        )

        _topics_train_i, _ = topic_model_i.fit_transform(fold_train_docs, embeddings=fold_train_embeddings)
        val_topics_i, _ = topic_model_i.transform(fold_val_docs, embeddings=fold_val_embeddings)

        topic_words_i = [
            [word for word, _ in topic_model_i.get_topic(t)]
            for t in topic_model_i.get_topics().keys()
            if t != -1 and topic_model_i.get_topic(t) is not None
        ]

        has_enough_for_cv = (
            len(processed_val_fold) >= 20 and
            len(id2word_fold) >= 30 and
            len(topic_words_i) >= 2
        )
        cv_val_i = sanitize_metric(
            coherence_score(topic_words_i, processed_val_fold, id2word_fold, 'c_v', jitter=COHERENCE_EPS),
            fallback=COHERENCE_EPS
        ) if has_enough_for_cv else np.nan

        sil_i = sanitize_metric(
            safe_silhouette_score(fold_val_embeddings, val_topics_i, outlier_label=-1, metric='cosine'),
            fallback=0.0
        )
        div_i = sanitize_metric(topic_diversity(topic_words_i), fallback=0.0)
        outlier_i = sanitize_metric(
            float(np.mean(np.array(val_topics_i) == -1)) if len(val_topics_i) > 0 else np.nan,
            fallback=1.0
        )
        singleton_i = sanitize_metric(singleton_ratio_from_labels(val_topics_i, outlier_label=-1), fallback=1.0)
        n_topics_i = int(sum(1 for t in topic_model_i.get_topics().keys() if int(t) != -1))

        elapsed = time.perf_counter() - start

        cv_rows.append({
            'trial': trial,
            'fold': fold['fold'],
            'random_seed': RANDOM_SEED,
            'n_neighbors': n_neighbors,
            'n_components': n_components,
            'min_cluster_size': min_cluster_size,
            'min_samples': min_samples,
            'ngram_range': str(ngram_range),
            'n_topics': n_topics_i,
            'cv_val': cv_val_i,
            'val_silhouette': sil_i,
            'topic_diversity': div_i,
            'val_outlier_ratio': outlier_i,
            'val_singleton_ratio': singleton_i,
            'fit_seconds': elapsed
        })

cv_results = pd.DataFrame(cv_rows)

agg_cols = ['cv_val', 'val_silhouette', 'topic_diversity', 'val_outlier_ratio', 'val_singleton_ratio', 'n_topics', 'fit_seconds']
tuning_results = cv_results.groupby(
    ['n_neighbors', 'n_components', 'min_cluster_size', 'min_samples', 'ngram_range'],
    as_index=False
)[agg_cols].mean()

def minmax_norm(series, higher_is_better=True):
    s = pd.to_numeric(series, errors='coerce').replace([np.inf, -np.inf], np.nan)
    if s.dropna().empty:
        return pd.Series(0.0, index=s.index)
    filled = s.fillna(s.min() if higher_is_better else s.max())
    min_v, max_v = float(filled.min()), float(filled.max())
    if max_v == min_v:
        return pd.Series(0.0, index=filled.index)
    norm = (filled - min_v) / (max_v - min_v)
    return norm

tuning_results['cv_val_norm'] = minmax_norm(tuning_results['cv_val'], higher_is_better=True)
tuning_results['val_silhouette_norm'] = minmax_norm(tuning_results['val_silhouette'], higher_is_better=True)
tuning_results['topic_diversity_norm'] = minmax_norm(tuning_results['topic_diversity'], higher_is_better=True)
tuning_results['val_outlier_ratio_norm'] = minmax_norm(tuning_results['val_outlier_ratio'], higher_is_better=False)
tuning_results['val_singleton_ratio_norm'] = minmax_norm(tuning_results['val_singleton_ratio'], higher_is_better=False)

# Penalize too few topics
tuning_results['topic_count_penalty'] = np.where(tuning_results['n_topics'] < 3, 0.25, 0.0)

# Composite for model selection (fixed seed), emphasizing cleaner cluster structure
tuning_results['composite_score'] = (
    0.30 * tuning_results['cv_val_norm'] +
    0.20 * tuning_results['val_silhouette_norm'] +
    0.20 * tuning_results['topic_diversity_norm'] +
    0.15 * tuning_results['val_outlier_ratio_norm'] +
    0.15 * tuning_results['val_singleton_ratio_norm'] -
    tuning_results['topic_count_penalty']
)

tuning_results = tuning_results.sort_values('composite_score', ascending=False).reset_index(drop=True)
best_params = tuning_results.iloc[0].to_dict()

# Bucket-wise (fold-wise) best params to inspect drift over time
fold_agg = cv_results.groupby(
    ['fold', 'n_neighbors', 'n_components', 'min_cluster_size', 'min_samples', 'ngram_range'],
    as_index=False
)[['cv_val', 'val_silhouette', 'topic_diversity', 'val_outlier_ratio', 'val_singleton_ratio', 'n_topics']].mean()

fold_agg['cv_val_norm'] = fold_agg.groupby('fold')['cv_val'].transform(lambda x: minmax_norm(x, True))
fold_agg['val_silhouette_norm'] = fold_agg.groupby('fold')['val_silhouette'].transform(lambda x: minmax_norm(x, True))
fold_agg['topic_diversity_norm'] = fold_agg.groupby('fold')['topic_diversity'].transform(lambda x: minmax_norm(x, True))
fold_agg['val_outlier_ratio_norm'] = fold_agg.groupby('fold')['val_outlier_ratio'].transform(lambda x: minmax_norm(x, False))
fold_agg['val_singleton_ratio_norm'] = fold_agg.groupby('fold')['val_singleton_ratio'].transform(lambda x: minmax_norm(x, False))
fold_agg['topic_count_penalty'] = np.where(fold_agg['n_topics'] < 3, 0.25, 0.0)
fold_agg['fold_score'] = (
    0.30 * fold_agg['cv_val_norm'] +
    0.20 * fold_agg['val_silhouette_norm'] +
    0.20 * fold_agg['topic_diversity_norm'] +
    0.15 * fold_agg['val_outlier_ratio_norm'] +
    0.15 * fold_agg['val_singleton_ratio_norm'] -
    fold_agg['topic_count_penalty']
)

best_per_fold = (
    fold_agg.sort_values(['fold', 'fold_score'], ascending=[True, False])
            .groupby('fold', as_index=False)
            .head(1)
            .reset_index(drop=True)
)

print('\nTop configurations (overall):')
display(tuning_results.head(10))

print('\nBest params by time bucket (fold):')
display(best_per_fold[['fold', 'n_neighbors', 'n_components', 'min_cluster_size', 'min_samples', 'ngram_range', 'cv_val', 'val_silhouette', 'val_outlier_ratio', 'val_singleton_ratio', 'fold_score']])

print('\nAverage fit time per model run (seconds):', round(float(cv_results['fit_seconds'].mean()), 2))
print('Best overall configuration selected from rolling CV (fixed-seed model selection):')
display(pd.DataFrame([best_params]))

Batches: 100%|██████████| 50/50 [00:00<00:00, 62.77it/s]



Generated 3 rolling folds (train+val pool only).
Fold 1: train 2024-09-17 00:00:00 -> 2025-07-03 00:00:00 | val 2025-07-05 00:00:00 -> 2025-09-24 00:00:00
Fold 2: train 2024-09-17 00:00:00 -> 2025-09-24 00:00:00 | val 2025-09-26 00:00:00 -> 2025-12-15 00:00:00
Fold 3: train 2024-09-17 00:00:00 -> 2025-12-15 00:00:00 | val 2025-12-17 00:00:00 -> 2026-03-10 00:00:00
Running 144 parameter combinations x 3 folds (fixed seed=42)
[Trial 1/144] n_neighbors=15, n_components=5, min_cluster_size=5, min_samples=1, ngram_range=(1, 1)
[Trial 2/144] n_neighbors=15, n_components=5, min_cluster_size=5, min_samples=1, ngram_range=(1, 2)
[Trial 2/144] n_neighbors=15, n_components=5, min_cluster_size=5, min_samples=1, ngram_range=(1, 2)
[Trial 3/144] n_neighbors=15, n_components=5, min_cluster_size=5, min_samples=3, ngram_range=(1, 1)
[Trial 3/144] n_neighbors=15, n_components=5, min_cluster_size=5, min_samples=3, ngram_range=(1, 1)
[Trial 4/144] n_neighbors=15, n_components=5, min_cluster_size=5, min_sa

,n_neighbors,n_components,min_cluster_size,min_samples,ngram_range,cv_val,val_silhouette,topic_diversity,val_outlier_ratio,val_singleton_ratio,n_topics,fit_seconds,cv_val_norm,val_silhouette_norm,topic_diversity_norm,val_outlier_ratio_norm,val_singleton_ratio_norm,topic_count_penalty,composite_score
0,50,10,5,5,"(1, 2)",0.543919,0.385527,0.927317,0.832926,0.312458,45.666667,12.042816,0.760794,0.902343,0.768026,0.961013,0.869404,0.0,0.836875
1,50,5,5,3,"(1, 2)",0.531726,0.421156,0.902653,0.853440,0.359393,63.333333,12.211773,0.696199,1.000000,0.538287,1.000000,1.000000,0.0,0.816517
2,50,10,8,5,"(1, 2)",0.509493,0.334252,0.913680,0.822634,0.306436,37.000000,11.949860,0.578419,0.761798,0.641002,0.941451,0.852649,0.0,0.723201
3,50,10,5,3,"(1, 2)",0.523197,0.379900,0.910527,0.834441,0.197747,60.333333,12.157043,0.651015,0.886919,0.611638,0.963890,0.550224,0.0,0.722133
4,50,5,5,5,"(1, 2)",0.516601,0.354599,0.917810,0.838986,0.179559,46.333333,12.076739,0.616072,0.817570,0.679469,0.972529,0.499618,0.0,0.705052
5,50,10,10,5,"(1, 2)",0.508700,0.278047,0.925313,0.802212,0.253788,30.333333,11.874969,0.574217,0.607742,0.749358,0.902637,0.706156,0.0,0.685004
6,50,5,8,3,"(1, 2)",0.506672,0.312027,0.913108,0.820464,0.217143,42.000000,11.982552,0.563474,0.700880,0.635673,0.937327,0.604193,0.0,0.667581
7,30,10,5,5,"(1, 2)",0.542100,0.267173,0.917147,0.740355,0.156460,49.000000,11.930133,0.751161,0.577937,0.673295,0.785076,0.435346,0.0,0.658658
8,50,5,5,1,"(1, 2)",0.504845,0.324327,0.908040,0.805422,0.214620,80.333333,12.196955,0.553792,0.734593,0.588467,0.908740,0.597172,0.0,0.656636
9,50,5,8,5,"(1, 2)",0.509918,0.315876,0.912612,0.836760,0.121693,37.666667,12.047716,0.580670,0.711430,0.631055,0.968298,0.338607,0.0,0.638734



Best params by time bucket (fold):


,fold,n_neighbors,n_components,min_cluster_size,min_samples,ngram_range,cv_val,val_silhouette,val_outlier_ratio,val_singleton_ratio,fold_score
0,1,50,10,5,5,"(1, 2)",0.497529,0.511788,0.934297,0.600000,0.746943
1,2,50,5,5,3,"(1, 2)",0.573664,0.435903,0.837875,0.346154,0.842452
2,3,50,5,5,3,"(1, 2)",0.560880,0.275412,0.764496,0.176471,0.800011



Average fit time per model run (seconds): 11.73
Best overall configuration selected from rolling CV (fixed-seed model selection):


,n_neighbors,n_components,min_cluster_size,min_samples,ngram_range,cv_val,val_silhouette,topic_diversity,val_outlier_ratio,val_singleton_ratio,n_topics,fit_seconds,cv_val_norm,val_silhouette_norm,topic_diversity_norm,val_outlier_ratio_norm,val_singleton_ratio_norm,topic_count_penalty,composite_score
0,50,10,5,5,"(1, 2)",0.543919,0.385527,0.927317,0.832926,0.312458,45.666667,12.042816,0.760794,0.902343,0.768026,0.961013,0.869404,0.0,0.836875


Final model and test evaluation with multi-seed variability analysis (single-touch per seed on test split)

In [35]:
from ast import literal_eval

# Refit best hyperparameter configuration on train+val, then evaluate variability across random seeds on test
EVAL_SEEDS = [42, 7, 123, 2024, 99]

train_val_docs = train_docs + val_docs
train_val_embeddings = sentence_model.encode(
    train_val_docs,
    show_progress_bar=True,
    batch_size=64,
    convert_to_numpy=True
)
test_embeddings = sentence_model.encode(
    test_docs,
    show_progress_bar=True,
    batch_size=64,
    convert_to_numpy=True
)

processed_train_val_docs = preprocess_documents(train_val_docs)
processed_test_docs = preprocess_documents(test_docs)
id2word_train_val = Dictionary(processed_train_val_docs)

best_ngram = literal_eval(best_params['ngram_range'])
seed_rows = []

for seed in EVAL_SEEDS:
    print(f'Running final fit/eval for seed={seed}...')

    best_umap = UMAP(
        n_neighbors=int(best_params['n_neighbors']),
        n_components=int(best_params['n_components']),
        metric='cosine',
        random_state=seed
    )
    best_hdbscan = HDBSCAN(
        min_cluster_size=int(best_params['min_cluster_size']),
        min_samples=int(best_params['min_samples']),
        metric='euclidean',
        cluster_selection_method='eom',
        prediction_data=True
    )
    best_vectorizer = OnlineCountVectorizer(stop_words=model_stopwords, ngram_range=best_ngram)

    best_topic_model = BERTopic(
        embedding_model=sentence_model,
        umap_model=best_umap,
        hdbscan_model=best_hdbscan,
        vectorizer_model=best_vectorizer,
        calculate_probabilities=True,
        verbose=False
    )

    _topics_train_val, _ = best_topic_model.fit_transform(train_val_docs, embeddings=train_val_embeddings)
    test_topics, _ = best_topic_model.transform(test_docs, embeddings=test_embeddings)

    best_topic_words = [
        [word for word, _ in best_topic_model.get_topic(t)]
        for t in best_topic_model.get_topics().keys()
        if t != -1 and best_topic_model.get_topic(t) is not None
    ]

    cv_test = coherence_score(best_topic_words, processed_test_docs, id2word_train_val, 'c_v', jitter=COHERENCE_EPS)
    div_test = topic_diversity(best_topic_words)
    sil_test = sanitize_metric(
        safe_silhouette_score(test_embeddings, test_topics, outlier_label=-1, metric='cosine'),
        fallback=0.0
    )
    test_outlier_ratio = float(np.mean(np.array(test_topics) == -1)) if len(test_topics) > 0 else np.nan
    test_topic_count = int(sum(1 for t in best_topic_model.get_topics().keys() if int(t) != -1))

    seed_rows.append({
        'seed': seed,
        'cv_test': cv_test,
        'test_silhouette': sil_test,
        'topic_diversity_test': div_test,
        'test_outlier_ratio': test_outlier_ratio,
        'test_topic_count': test_topic_count
    })

seed_results = pd.DataFrame(seed_rows)

print('\nPer-seed test metrics:')
display(seed_results)

summary_stats = pd.DataFrame([{
    'cv_test_mean': seed_results['cv_test'].mean(skipna=True),
    'cv_test_std': seed_results['cv_test'].std(skipna=True),
    'test_silhouette_mean': seed_results['test_silhouette'].mean(skipna=True),
    'test_silhouette_std': seed_results['test_silhouette'].std(skipna=True),
    'topic_diversity_mean': seed_results['topic_diversity_test'].mean(skipna=True),
    'topic_diversity_std': seed_results['topic_diversity_test'].std(skipna=True),
    'test_outlier_ratio_mean': seed_results['test_outlier_ratio'].mean(skipna=True),
    'test_outlier_ratio_std': seed_results['test_outlier_ratio'].std(skipna=True),
    'test_topic_count_mean': seed_results['test_topic_count'].mean(skipna=True),
    'test_topic_count_std': seed_results['test_topic_count'].std(skipna=True)
}])

print('\nFinal test metrics variability across random seeds:')
display(summary_stats)

def format_mean_std(mean_value, std_value, digits=4):
    if not np.isfinite(mean_value):
        return 'NA'
    if not np.isfinite(std_value):
        return f"{mean_value:.{digits}f} ± NA"
    return f"{mean_value:.{digits}f} ± {std_value:.{digits}f}"

thesis_report = pd.DataFrame([{
    'Model': 'BERTopic (best CV config)',
    'Seeds (n)': len(EVAL_SEEDS),
    'Coherence C_v (test)': format_mean_std(summary_stats.at[0, 'cv_test_mean'], summary_stats.at[0, 'cv_test_std'], digits=4),
    'Silhouette (test, cosine)': format_mean_std(summary_stats.at[0, 'test_silhouette_mean'], summary_stats.at[0, 'test_silhouette_std'], digits=4),
    'Topic Diversity (test)': format_mean_std(summary_stats.at[0, 'topic_diversity_mean'], summary_stats.at[0, 'topic_diversity_std'], digits=4),
    'Outlier Ratio (test)': format_mean_std(summary_stats.at[0, 'test_outlier_ratio_mean'], summary_stats.at[0, 'test_outlier_ratio_std'], digits=4),
    'Topic Count (test)': format_mean_std(summary_stats.at[0, 'test_topic_count_mean'], summary_stats.at[0, 'test_topic_count_std'], digits=2)
}])

print('\nThesis-ready reporting table (mean ± std across seeds):')
display(thesis_report)

Batches: 100%|██████████| 19/19 [00:00<00:00, 118.75it/s]



Running final fit/eval for seed=42...
Running final fit/eval for seed=7...
Running final fit/eval for seed=7...
Running final fit/eval for seed=123...
Running final fit/eval for seed=123...
Running final fit/eval for seed=2024...
Running final fit/eval for seed=2024...
Running final fit/eval for seed=99...
Running final fit/eval for seed=99...

Per-seed test metrics:

Per-seed test metrics:


,seed,cv_test,test_silhouette,topic_diversity_test,test_outlier_ratio,test_topic_count
0,42,0.444442,0.155670,0.891753,0.688312,97
1,7,0.482140,0.161408,0.884375,0.698701,96
2,123,0.486049,0.164328,0.898925,0.712554,93
3,2024,0.450734,0.151827,0.890722,0.716017,97
4,99,0.473996,0.157535,0.897849,0.654545,93



Final test metrics variability across random seeds:


,cv_test_mean,cv_test_std,test_silhouette_mean,test_silhouette_std,topic_diversity_mean,topic_diversity_std,test_outlier_ratio_mean,test_outlier_ratio_std,test_topic_count_mean,test_topic_count_std
0,0.467472,0.018797,0.158153,0.004881,0.892725,0.005903,0.694026,0.024699,95.2,2.04939



Thesis-ready reporting table (mean ± std across seeds):


,Model,Seeds (n),Coherence C_v (test),"Silhouette (test, cosine)",Topic Diversity (test),Outlier Ratio (test),Topic Count (test)
0,BERTopic (best CV config),5,0.4675 ± 0.0188,0.1582 ± 0.0049,0.8927 ± 0.0059,0.6940 ± 0.0247,95.20 ± 2.05


In [36]:
import plotly.express as px

# Diagnostics use the currently available final test assignment (`test_topics`) and embeddings (`test_embeddings`).
if 'test_topics' not in globals() or 'test_embeddings' not in globals():
    raise RuntimeError('Please run Cell 24 first so `test_topics` and `test_embeddings` are available.')

labels = np.asarray(test_topics)
emb = np.asarray(test_embeddings)

if labels.shape[0] != emb.shape[0]:
    raise ValueError(f'Mismatch: labels={labels.shape[0]} vs embeddings={emb.shape[0]}')

is_outlier = labels == -1
outlier_ratio_now = float(np.mean(is_outlier)) if len(labels) > 0 else np.nan

valid_labels = labels[~is_outlier]
if len(valid_labels) > 0:
    cluster_size_series = pd.Series(valid_labels).value_counts().sort_values(ascending=False)
else:
    cluster_size_series = pd.Series(dtype='int64')

n_valid_clusters_now = int(cluster_size_series.shape[0])
singleton_topics_now = set(cluster_size_series[cluster_size_series == 1].index.tolist())
n_singletons_now = int(len(singleton_topics_now))

print('Final-clustering diagnostics (current test assignment):')
print(f'  Documents: {len(labels)}')
print(f'  Outlier ratio (-1): {outlier_ratio_now:.4f}')
print(f'  Valid clusters (excluding -1): {n_valid_clusters_now}')
print(f'  Singleton clusters: {n_singletons_now}')

if n_valid_clusters_now == 0:
    print('⚠️ No valid clusters found (all points are outliers).')
elif n_valid_clusters_now == 1:
    print('⚠️ Only one valid cluster found. Silhouette is undefined by design.')
if n_singletons_now > 0:
    print('⚠️ Singleton clusters present. Silhouette excludes them in our robust function.')

# 1) Cluster size distribution (excluding outliers)
if n_valid_clusters_now > 0:
    cluster_size_df = (
        cluster_size_series
        .rename_axis('Topic')
        .reset_index(name='Count')
        .sort_values('Count', ascending=False)
    )
    cluster_size_df['Topic'] = cluster_size_df['Topic'].astype(int)

    fig_sizes = px.bar(
        cluster_size_df,
        x='Topic',
        y='Count',
        title='Final Test Cluster Sizes (excluding outliers)',
        labels={'Topic': 'Topic ID', 'Count': 'Number of Documents'}
    )
    fig_sizes.show()
else:
    print('Cluster-size plot skipped (no valid clusters).')

# 2) 2D embedding view colored by diagnostic point type
# Recompute a 2D projection from test embeddings for visual inspection.
umap_viz = UMAP(n_neighbors=15, n_components=2, metric='cosine', random_state=42)
emb_2d = umap_viz.fit_transform(emb)

point_type = np.where(
    labels == -1,
    'Outlier (-1)',
    np.where(np.isin(labels, list(singleton_topics_now)), 'Singleton cluster', 'Valid cluster')
)

viz_df = pd.DataFrame({
    'x': emb_2d[:, 0],
    'y': emb_2d[:, 1],
    'topic': labels.astype(int),
    'point_type': point_type
})

fig_diag = px.scatter(
    viz_df,
    x='x',
    y='y',
    color='point_type',
    hover_data=['topic'],
    title='Final Test Embedding Diagnostics (Outliers / Singleton / Valid)'
)
fig_diag.update_traces(marker=dict(size=6, opacity=0.7))
fig_diag.show()

# 3) Seed-level context from the multi-seed run
if 'seed_results' in globals() and isinstance(seed_results, pd.DataFrame) and not seed_results.empty:
    seed_plot_df = seed_results.copy()

    fig_seed_outliers = px.bar(
        seed_plot_df,
        x='seed',
        y='test_outlier_ratio',
        title='Outlier Ratio by Seed',
        labels={'seed': 'Seed', 'test_outlier_ratio': 'Outlier Ratio'}
    )
    fig_seed_outliers.show()

    fig_seed_topics = px.bar(
        seed_plot_df,
        x='seed',
        y='test_topic_count',
        title='Valid Topic Count by Seed',
        labels={'seed': 'Seed', 'test_topic_count': 'Topic Count (excluding -1)'}
    )
    fig_seed_topics.show()
else:
    print('Seed-level plots skipped (run Cell 24 first to populate `seed_results`).')

Final-clustering diagnostics (current test assignment):
  Documents: 1155
  Outlier ratio (-1): 0.6545
  Valid clusters (excluding -1): 45
  Singleton clusters: 15
⚠️ Singleton clusters present. Silhouette excludes them in our robust function.


Cluster diagnostics and visual sanity checks (outliers, valid clusters, singleton clusters)